# Análisis de correlaciones

**PP2 · ITSE** — Input: `viviendas_procesadas.csv`

Relaciones entre variables con hipótesis concretas. Cada análisis parte de una pregunta de negocio y cierra con una interpretación accionable para el ministerio.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install scipy seaborn -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
DRIVE = Path('/content/drive/MyDrive/PP2')
Path('img').mkdir(exist_ok=True)

df = pd.read_csv(DRIVE / 'viviendas_procesadas.csv')
col_avance = 'avance_obra' if 'avance_obra' in df.columns else 'avanceObra'
col_tipo   = 'tipo_vivienda' if 'tipo_vivienda' in df.columns else 'tipoVivienda'
col_dorm   = 'cant_dormitorios' if 'cant_dormitorios' in df.columns else 'cantDormitorios'
print(f'Registros: {len(df)} | Columnas: {df.shape[1]}')

In [ ]:
COL_CR = {'Inclusion':'#22c55e','Otro':'#3b82f6','Exclusion':'#ef4444'}
COL_E  = {'Iniciada':'#f59e0b','Avanzada':'#3b82f6','Finalizada':'#22c55e','Adjudicada':'#6366f1'}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Avance promedio por criterio × tipo: cruce entre la macrocategoría del programa y el tipo de vivienda
av_ct = (df.groupby(['criterio', col_tipo])[col_avance].mean()
           .unstack().reindex(['Inclusion','Otro','Exclusion']))
av_ct.plot(kind='bar', ax=axes[0], color=['#3b82f6','#22c55e','#f59e0b'], alpha=0.85)
axes[0].set_title('Avance promedio por criterio y tipo de vivienda')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Tipo', fontsize=8)

# Distribución de estados por criterio (barras apiladas al 100%): ¿los Exclusión finalizan menos?
estado_crit = (df.groupby(['criterio','estado']).size()
                 .unstack(fill_value=0)
                 .reindex(['Inclusion','Otro','Exclusion'])
                 .pipe(lambda d: d.div(d.sum(axis=1), axis=0) * 100))
estado_crit.plot(kind='bar', stacked=True, ax=axes[1],
                 color=[COL_E.get(c,'#94a3b8') for c in estado_crit.columns], alpha=0.85)
axes[1].set_title('Distribución de estados por criterio (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Estado', fontsize=8, loc='lower right')

plt.tight_layout(); plt.savefig('img/03_criterio_estado.png', bbox_inches='tight'); plt.show()

### Hipótesis: las viviendas rurales tienen menor avance promedio

El acceso al terreno y la provisión de materiales suele ser más difícil en zonas rurales, lo que alargaría los tiempos de obra. Si el test ANOVA confirma que la diferencia entre tipos es estadísticamente significativa, el ministerio debería ajustar los plazos contractuales para obras rurales o asignarles mayor soporte logístico.

In [ ]:
col_tipo_map = {'Urbana':'#3b82f6','Rural':'#22c55e','Económica':'#f59e0b'}
tipos = df[col_tipo].dropna().unique()
PLAZO = 90   # plazo contractual de construcción (días)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter días activa vs avance por tipo: ubica cada vivienda en el espacio tiempo-avance.
# Las líneas punteadas marcan la zona de riesgo: pasó el plazo de 90 días con avance < 80%.
for tipo in tipos:
    sub = df[df[col_tipo]==tipo]
    axes[0].scatter(sub['dias_activa'], sub[col_avance],
                    alpha=0.25, s=12, color=col_tipo_map.get(tipo,'#94a3b8'), label=tipo)
axes[0].axvline(PLAZO, color='#64748b', linestyle='--', alpha=0.5)
axes[0].axhline(80,  color='#64748b', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Días activa'); axes[0].set_ylabel('Avance (%)')
axes[0].set_title('Días activa vs. avance por tipo'); axes[0].legend(fontsize=9)

# Barra de promedios con desvío estándar
st_tipo = df.groupby(col_tipo)[col_avance].agg(['mean','std']).reset_index()
axes[1].bar(st_tipo[col_tipo], st_tipo['mean'], yerr=st_tipo['std'], capsize=5,
            color=[col_tipo_map.get(t,'#94a3b8') for t in st_tipo[col_tipo]], alpha=0.85)
for i, row in st_tipo.iterrows():
    axes[1].text(i, row['mean']+row['std']+2, f"{row['mean']:.1f}%", ha='center', fontsize=10)
axes[1].set_title('Avance promedio ± desvío por tipo')

plt.tight_layout(); plt.savefig('img/03_avance_tipo.png', bbox_inches='tight'); plt.show()

# ANOVA en lugar de t-test porque comparamos 3 grupos (Urbana / Rural / Económica).
# El t-test solo compara 2; aplicarlo múltiples veces sobre el mismo conjunto inflaría el error tipo I.
data_g = [df[df[col_tipo]==t][col_avance].dropna() for t in tipos]
f_stat, p = stats.f_oneway(*data_g)
print(f'ANOVA — F={f_stat:.2f}, p={p:.4f} — {"Diferencia significativa" if p < 0.05 else "Sin diferencia"}')

### Criterio × clasificación: dos vistas del mismo dato

**Criterio** (Inclusion / Exclusion / Otro) agrupa los 15 códigos en 3 macrocategorías. Permite comparar tendencias de alto nivel entre tipos de intervención.

**Clasificación** muestra los 15 códigos individuales, útil para ajustar plazos contractuales según la complejidad específica de cada tipo de vivienda.

Combinadas, permiten detectar si las obras clasificadas como Exclusión que igualmente ingresaron al programa muestran duraciones o tasas de riesgo distintas a las de Inclusión.

In [ ]:
COL_CR = {'Inclusion':'#22c55e','Otro':'#3b82f6','Exclusion':'#ef4444'}
PLAZO = 90   # plazo contractual de construcción (días)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

st_crit = df.groupby('criterio')['dias_activa'].agg(['mean','std']).reindex(['Inclusion','Otro','Exclusion'])
axes[0].bar(st_crit.index, st_crit['mean'], yerr=st_crit['std'], capsize=5,
            color=[COL_CR[c] for c in st_crit.index], alpha=0.85)
for i, (c, row) in enumerate(st_crit.iterrows()):
    axes[0].text(i, row['mean']+row['std']+5, f"{row['mean']:.0f}d", ha='center', fontsize=10)
axes[0].axhline(PLAZO, color='#ef4444', linestyle='--', alpha=0.5, label=f'{PLAZO} días (plazo)')
axes[0].set_title('Duración promedio por criterio'); axes[0].legend(fontsize=9)

st_c = df.groupby('clasificacion')['dias_activa'].mean().sort_values()
axes[1].barh(st_c.index, st_c.values, color='#6366f1', alpha=0.8)
axes[1].axvline(PLAZO, color='#ef4444', linestyle='--', alpha=0.5)
axes[1].set_title('Duración promedio por clasificación (días)')

plt.tight_layout(); plt.savefig('img/03_duracion.png', bbox_inches='tight'); plt.show()

### Análisis de cohortes: ¿el programa mejora o arrastra demoras?

Agrupar las obras por su año de inicio permite ver la **maduración del programa**: una cohorte vieja (2022) debería estar mayormente terminada; una reciente (2025-2026), todavía activa.

La señal de alerta es una **cohorte vieja con alta proporción de obras aún activas o en riesgo** — significa que esas obras llevan años sin cerrarse. Hay que leer las cohortes recientes con cuidado: su baja tasa de finalización es esperable porque todavía están dentro de su plazo.

In [ ]:
# Cohortes por año de inicio + avance por departamento
C = {'base':'#4f46e5','ok':'#10b981','medio':'#f59e0b','alerta':'#f43f5e','neutro':'#94a3b8'}
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Análisis de cohortes: de las obras iniciadas cada año, en qué estado están hoy.
# Las cohortes viejas deberían estar mayormente terminadas; las recientes, activas.
# Si una cohorte vieja todavía tiene muchas activas/en riesgo, el programa arrastra demoras.
coh = (df.groupby('anio_inicio')
         .agg(terminadas=('estado', lambda s: s.isin(['Finalizada','Adjudicada']).mean()*100),
              riesgo    =('nivel_riesgo', lambda s: (s=='alto').mean()*100))
         .assign(activas_ok=lambda d: 100 - d['terminadas'] - d['riesgo']))
coh = coh[coh.index >= 2022]

axes[0].bar(coh.index.astype(str), coh['terminadas'], color=C['ok'], label='Terminadas', alpha=0.9)
axes[0].bar(coh.index.astype(str), coh['activas_ok'], bottom=coh['terminadas'],
            color=C['base'], label='Activas sin riesgo', alpha=0.9)
axes[0].bar(coh.index.astype(str), coh['riesgo'], bottom=coh['terminadas']+coh['activas_ok'],
            color=C['alerta'], label='En riesgo alto', alpha=0.9)
axes[0].set_title('Cohortes: estado actual según año de inicio (%)')
axes[0].set_ylabel('% de la cohorte'); axes[0].legend(fontsize=8, loc='lower left')

# Avance promedio por departamento (rojo = tiene obras en riesgo alto)
av_dep = (df.groupby('departamento')
            .agg(avance=(col_avance,'mean'), riesgo=('nivel_riesgo', lambda x: (x=='alto').sum()))
            .sort_values('avance'))
axes[1].barh(av_dep.index, av_dep['avance'],
             color=[C['alerta'] if r > 0 else C['base'] for r in av_dep['riesgo']], alpha=0.85)
axes[1].axvline(df[col_avance].mean(), color=C['neutro'], linestyle='--', label='Promedio')
axes[1].set_title('Avance promedio por departamento'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('img/03_cohortes_depto.png', bbox_inches='tight'); plt.show()
print(coh.round(1).to_string())

### Riesgo por criterio y clasificación

Como análisis de soporte: ¿varía la tasa de riesgo según el tipo de vivienda?

El primer gráfico agrupa por criterio (macro), el segundo desagrega por los 15 códigos. No es el hallazgo más crítico del EDA, pero contextualiza desde qué tipo de viviendas proviene el riesgo y si hay patrones en la selección de beneficiarios.

In [ ]:
COL_CR = {'Inclusion':'#22c55e','Otro':'#3b82f6','Exclusion':'#ef4444'}
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# % riesgo alto por criterio
riesgo_cr = (df.groupby('criterio')['nivel_riesgo']
               .value_counts(normalize=True).mul(100).round(1)
               .rename('pct').reset_index())
riesgo_cr = riesgo_cr[riesgo_cr['nivel_riesgo']=='alto']
axes[0].bar(riesgo_cr['criterio'], riesgo_cr['pct'],
            color=[COL_CR.get(c,'#94a3b8') for c in riesgo_cr['criterio']], alpha=0.85)
for i, row in riesgo_cr.reset_index(drop=True).iterrows():
    axes[0].text(i, row['pct']+0.5, f"{row['pct']:.1f}%", ha='center', fontsize=11)
axes[0].set_title('% de obras en riesgo alto por criterio')
axes[0].set_ylabel('% riesgo alto')

# % riesgo alto por clasificación (los 15 códigos) — solo obras activas
# Se filtra por activas porque las terminadas no pueden estar "en riesgo de no cerrar"
risk_c = (df[df['estado'].isin(['Iniciada','Avanzada'])]
          .groupby('clasificacion')
          .agg(total=(col_avance,'count'),
               alto=('nivel_riesgo', lambda x: (x=='alto').sum()))
          .assign(pct=lambda d: (d['alto']/d['total']*100).round(1))
          .sort_values('pct', ascending=True))
col_bars = ['#ef4444' if v>=40 else '#f59e0b' if v>=20 else '#22c55e' for v in risk_c['pct']]
axes[1].barh(risk_c.index, risk_c['pct'], color=col_bars, alpha=0.85)
axes[1].bar_label(axes[1].containers[0], fmt='%.0f%%', padding=3, fontsize=9)
axes[1].set_title('% riesgo alto por clasificación (obras activas)')
axes[1].set_xlabel('% riesgo alto')

plt.tight_layout(); plt.savefig('img/03_riesgo_criterio_clasif.png', bbox_inches='tight'); plt.show()
r, p = stats.pearsonr(df['criterio_enc'].fillna(0), df[col_avance])
print(f'Correlación criterio–avance: r={r:.3f}, p={p:.4f}')